# Statistical Tests

Beyond the core Mann-Whitney U hypothesis test , these are 8 follow-up tests that probe the same dataset from different angles.

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
from scipy.stats import (
    chi2_contingency, spearmanr, kruskal, wilcoxon
)
from sklearn.linear_model import LogisticRegression
import statsmodels.api as sm

pd.set_option('display.float_format', lambda x: f'{x:.4f}')

DB_PATH = "retail_behavior.db"
conn = sqlite3.connect(DB_PATH)

metrics_df = pd.read_sql("SELECT * FROM metrics_results", conn, parse_dates=["quarter_end_date", "results_date"])
shareholding = pd.read_sql("SELECT * FROM shareholding", conn, parse_dates=["quarter_end_date", "results_date"])
stocks = pd.read_sql("SELECT * FROM stocks", conn)

metrics_df = metrics_df.merge(stocks[["ticker", "sector"]], on="ticker", how="left")
shareholding = shareholding.merge(stocks[["ticker", "sector"]], on="ticker", how="left")

print(metrics_df.shape, shareholding.shape)
metrics_df.head()

(320, 9) (160, 8)


,ticker,quarter_end_date,results_date,window_days,volume_spike_ratio,price_range_pct,realized_vol,bucket,sector
0,HDFCBANK,2024-09-30,2024-10-23,3,0.9367,0.9164,0.0136,High,Finance
1,HDFCBANK,2024-09-30,2024-10-23,5,0.7823,0.8940,0.0130,High,Finance
2,HDFCBANK,2024-12-31,2025-01-28,3,1.2186,0.9228,0.0132,High,Finance
3,HDFCBANK,2024-12-31,2025-01-28,5,1.2293,1.0797,0.0115,High,Finance
4,HDFCBANK,2025-03-31,2025-04-19,3,1.3872,1.1121,0.0140,High,Finance


---
## Test 1 — Is retail% bucket associated with sector?

**Question:** Is the High/Low retail bucket simply more common in one sector than the other, independent of any erraticness effect?

**Why Chi-square test of independence:** both variables here are categorical (`bucket`: High/Low, `sector`: Finance / Manufacturing & Others). Chi-square tests whether two categorical variables are independent of each other. A within-sector median split should, by construction, roughly balance High/Low within each sector this test is a direct check that the construction worked as intended.

**H0:** Bucket (High/Low) is independent of sector — no association.
**H1:** Bucket is associated with sector — one sector has disproportionately more High (or Low) stocks.


In [ ]:
# Use one row per ticker-quarter (collapse window_days=3/5 duplicates) to avoid double-counting
unique_bucket = metrics_df[metrics_df["window_days"] == 3][["ticker", "quarter_end_date", "sector", "bucket"]]

contingency_sector_bucket = pd.crosstab(unique_bucket["sector"], unique_bucket["bucket"])
print(contingency_sector_bucket)

chi2, p, dof, expected = chi2_contingency(contingency_sector_bucket)
print(f"\nChi-square = {chi2:.4f}, dof = {dof}, p = {p:.4f}")
print("Expected counts under H0 (independence):")
print(pd.DataFrame(expected, index=contingency_sector_bucket.index, columns=contingency_sector_bucket.columns))

bucket                    High  Low
sector                             
Finance                     40   40
Manufacturing and Others    40   40

Chi-square = 0.0000, dof = 1, p = 1.0000
Expected counts under H0 (independence):
bucket                      High     Low
sector                                  
Finance                  40.0000 40.0000
Manufacturing and Others 40.0000 40.0000


**Reading the result:** p >= 0.05 -> fail to reject, bucket composition looks sector-independent.

## Test 2 — Does volume spike ratio correlate with retail% as a continuous variable?

**Question:** Is there a real monotonic relationship between retail ownership % and volume spike ratio, that the binary High/Low split might be flattening out?

**Why Spearman rank correlation, not Pearson:** Pearson assumes a linear relationship and is sensitive to outliers/skew. retail% and metric distributions are already **skewed**. Spearman works on ranks, not raw values, so it detects any monotonic relationship without being thrown off by outliers like IIFL's spike.

**H0:** No monotonic correlation between retail% and volume_spike_ratio (Spearman's rho = 0).
**H1:** A monotonic correlation exists (rho != 0).


In [ ]:
joined = metrics_df.merge(
    shareholding[["ticker", "quarter_end_date", "public_retail_pct"]],
    on=["ticker", "quarter_end_date"], how="left"
)

for window in [3, 5]:
    sub = joined[joined["window_days"] == window]
    print(f"--- window = {window} ---")
    for metric in ["volume_spike_ratio", "price_range_pct", "realized_vol"]:
        rho, p = spearmanr(sub["public_retail_pct"], sub[metric], nan_policy="omit")
        print(f"{metric}: rho = {rho:.4f}, p = {p:.4f}")
    print()

--- window = 3 ---
volume_spike_ratio: rho = -0.0487, p = 0.5412
price_range_pct: rho = -0.0818, p = 0.3039
realized_vol: rho = 0.0536, p = 0.5006

--- window = 5 ---
volume_spike_ratio: rho = -0.0050, p = 0.9503
price_range_pct: rho = -0.0188, p = 0.8131
realized_vol: rho = 0.1193, p = 0.1329



**Reading the result:** rho close to 0 and p >= 0.05 -> no monotonic relationship.

---
## Test 3 — Do sectors differ regardless of retail bucket?

**Question:** Does *sector* itself drive differences in erraticness metrics, separate from the retail-bucket effect?

**Why Kruskal-Wallis:** it's the non-parametric equivalent of one-way ANOVA for **more than 2 groups** — here we have 4 groups (Finance-High, Finance-Low, Manufacturing-High, Manufacturing-Low). Mann-Whitney only compares 2 groups at a time; Kruskal-Wallis tests whether *any* of the 4 groups differs from the others, in one test, without inflating false-positive risk the way running multiple pairwise Mann-Whitneys would.

**H0:** All 4 sector x bucket groups come from the same distribution (no group differs).
**H1:** At least one group differs from the others.


In [ ]:
metrics_df["sector_bucket"] = metrics_df["sector"] + " - " + metrics_df["bucket"]

for window in [3, 5]:
    sub = metrics_df[metrics_df["window_days"] == window]
    print(f"--- window = {window} ---")
    for metric in ["volume_spike_ratio", "price_range_pct", "realized_vol"]:
        groups = [g[metric].dropna().values for _, g in sub.groupby("sector_bucket")]
        labels = sub["sector_bucket"].unique()
        stat, p = kruskal(*groups)
        print(f"{metric}: H = {stat:.4f}, p = {p:.4f}  (groups: {list(labels)})")
    print()

--- window = 3 ---
volume_spike_ratio: H = 2.6348, p = 0.4514  (groups: ['Finance - High', 'Finance - Low', 'Manufacturing and Others - High', 'Manufacturing and Others - Low'])
price_range_pct: H = 4.5897, p = 0.2044  (groups: ['Finance - High', 'Finance - Low', 'Manufacturing and Others - High', 'Manufacturing and Others - Low'])
realized_vol: H = 4.9012, p = 0.1792  (groups: ['Finance - High', 'Finance - Low', 'Manufacturing and Others - High', 'Manufacturing and Others - Low'])

--- window = 5 ---
volume_spike_ratio: H = 1.5856, p = 0.6627  (groups: ['Finance - High', 'Finance - Low', 'Manufacturing and Others - High', 'Manufacturing and Others - Low'])
price_range_pct: H = 1.6057, p = 0.6581  (groups: ['Finance - High', 'Finance - Low', 'Manufacturing and Others - High', 'Manufacturing and Others - Low'])
realized_vol: H = 3.8090, p = 0.2828  (groups: ['Finance - High', 'Finance - Low', 'Manufacturing and Others - High', 'Manufacturing and Others - Low'])



**Reading the result:** p >= 0.05 -> no group (sector or bucket combination) stands out, strengthening the case that null result isn't being masked by a sector-level confound.

## Test 4 — Does realized_vol differ between the ±3d and ±5d windows for the same stock-quarter?

**Question:** Is the wider (±5d) window picking up materially different (likely more noise-diluted) volatility than the tighter ±3d window, for the *same* event?

**Why Wilcoxon signed-rank test, not another Mann-Whitney:** Mann-Whitney assumes the two samples are **independent**. Here they are not , the ±3d and ±5d rows for a given stock-quarter are the *same event* measured two ways, i.e. paired observations. Using Mann-Whitney on paired data wastes the pairing information and is the wrong test. Wilcoxon signed-rank is the non-parametric test for paired samples (the paired-data analogue of Mann-Whitney), correct here because your data is skewed (same reasoning as before) and the two values are linked by stock-quarter.

**H0:** The median difference between ±3d and ±5d realized_vol (for the same stock-quarter) is zero.

**H1:** The median difference is non-zero.


In [ ]:
paired = metrics_df.pivot_table(
    index=["ticker", "quarter_end_date"],
    columns="window_days",
    values=["volume_spike_ratio", "price_range_pct", "realized_vol"]
)
paired.columns = [f"{m}_{w}d" for m, w in paired.columns]
paired = paired.dropna()

for metric in ["volume_spike_ratio", "price_range_pct", "realized_vol"]:
    col3, col5 = f"{metric}_3d", f"{metric}_5d"
    stat, p = wilcoxon(paired[col3], paired[col5])
    diff_median = (paired[col3] - paired[col5]).median()
    print(f"{metric}: Wilcoxon stat = {stat:.4f}, p = {p:.4f}, median(±3d - ±5d) = {diff_median:.4f}")

volume_spike_ratio: Wilcoxon stat = 1694.0000, p = 0.0000, median(±3d - ±5d) = 0.1435
price_range_pct: Wilcoxon stat = 3528.0000, p = 0.0000, median(±3d - ±5d) = 0.0763
realized_vol: Wilcoxon stat = 4629.0000, p = 0.0020, median(±3d - ±5d) = 0.0011


**Reading the result:** p < 0.05 -> All 3 metrics show a statistically significant difference between ±3d and ±5d.

**What this means** : for the same stock-quarter, the ±3d window consistently produces a higher value than the ±5d window, on all 3 metrics.

A paired Wilcoxon signed-rank test confirms ±3d values are significantly higher than ±5d values for the same stock-quarter across all three metrics, consistent with the EDA finding that market reaction concentrates in the days immediately around results and dilutes as the window widens

---
## Test 5 — Do the 3 metrics jointly predict the bucket label?

**Question:** Even if no single metric significantly separates High vs Low on its own, do the 3 metrics *together* carry more predictive signal about bucket membership than any one does alone?

**Why logistic regression:** the outcome (`bucket`: High=1/Low=0) is binary.

**H0 (practically):** None of the 3 metrics' coefficients are significantly different from zero , they carry no predictive information about bucket.

**H1:** At least one metric's coefficient is significantly non-zero.


In [ ]:
import statsmodels.formula.api as smf

for window in [3, 5]:
    sub = metrics_df[metrics_df["window_days"] == window].copy()
    sub["bucket_bin"] = (sub["bucket"] == "High").astype(int)
    sub = sub.dropna(subset=["volume_spike_ratio", "price_range_pct", "realized_vol"])

    logit_model = smf.logit(
        "bucket_bin ~ volume_spike_ratio + price_range_pct + realized_vol",
        data=sub
    ).fit(disp=0)

    print(f"=== window = {window} ===")
    print(logit_model.summary())
    print()

=== window = 3 ===
                           Logit Regression Results                           
Dep. Variable:             bucket_bin   No. Observations:                  160
Model:                          Logit   Df Residuals:                      156
Method:                           MLE   Df Model:                            3
Date:                Thu, 08 Oct 2026   Pseudo R-squ.:                0.007466
Time:                        12:45:52   Log-Likelihood:                -110.08
converged:                       True   LL-Null:                       -110.90
Covariance Type:            nonrobust   LLR p-value:                    0.6468
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.2088      0.604      0.346      0.730      -0.975       1.392
volume_spike_ratio     0.1013      0.321      0.315      0.753      -0.529       0.731
p

**Reading the result:** looking at each coefficient's p-value , we can say None of the 3 metrics' coefficients are significantly different from zero

## Test 6 — Does realized_vol depend on all 4 ownership types jointly, not just public retail%?

**Question:** Once we have promoter%, FII%, and DII% ownership, does public retail% still matter for realized_vol or does its apparent effect change once institutional ownership is accounted for?

**Why multiple linear regression:**  have 4 continuous ownership-percentage predictors and 1 continuous outcome (realized_vol). Multiple regression lets isolate the effect of retail% *holding the other 3 ownership types constant*  something a simple two-group Mann-Whitney test on retail% alone cannot do, since it ignores promoter/FII/DII entirely.

**H0:** The coefficient on `public_retail_pct` is zero once promoter_pct, fii_pct, dii_pct are controlled for.

**H1:** The coefficient on `public_retail_pct` is non-zero, controlling for the other ownership types.


In [ ]:
ownership_joined = metrics_df.merge(
    shareholding[["ticker", "quarter_end_date", "promoter_pct", "fii_pct", "dii_pct", "public_retail_pct"]],
    on=["ticker", "quarter_end_date"], how="left"
)

for window in [3, 5]:
    sub = ownership_joined[ownership_joined["window_days"] == window].dropna(
        subset=["realized_vol", "promoter_pct", "fii_pct", "dii_pct", "public_retail_pct"]
    )
    X = sm.add_constant(sub[["promoter_pct", "fii_pct", "dii_pct", "public_retail_pct"]])
    y = sub["realized_vol"]
    ols_model = sm.OLS(y, X).fit()

    print(f"=== window = {window} ===")
    print(ols_model.summary())
    print()

=== window = 3 ===
                            OLS Regression Results                            
Dep. Variable:           realized_vol   R-squared:                       0.080
Model:                            OLS   Adj. R-squared:                  0.050
Method:                 Least Squares   F-statistic:                     2.688
Date:                Thu, 08 Oct 2026   Prob (F-statistic):             0.0344
Time:                        12:45:55   Log-Likelihood:                 390.11
No. Observations:                 128   AIC:                            -770.2
Df Residuals:                     123   BIC:                            -756.0
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const              

**Reading the result:** Fail to reject H0 for public_retail_pct — its coefficient is not significantly different from zero at either window even after controlling for the other 3 ownership types.

But DII (domestic institutional investor) ownership **has a real, negative, statistically significant relationship** with realized_vol, at both windows. Higher DII ownership is associated with lower volatility around results.

DII investors (mutual funds, insurance companies) are large, slow-moving, long-horizon holders — their presence tends to stabilize a stock's float and reduce the pool of shares available for reactive, short-term trading. **DII ownership, not retail ownership, predicts realized volatility around results.**

## Test 7 — Is there a time trend in retail% for the 3 "drifting" stocks?

**Question:** For IDFCFIRSTB, ICICIBANK, IDEA — the 3 stocks flagged as having retail% that changes over the sample period — is that drift actually a statistically detectable linear trend, or just quarter-to-quarter noise?

**Why linear regression on time:**  want to quantify a trend over ordered quarters for each stock individually. a direct, quantified answer to "how much is retail% moving per quarter", rather than the qualitative "this stock seems to be drifting" statement from the earlier EDA.

**H0 (per stock):** The slope of retail% over quarter index is zero (no trend).

**H1:** The slope is non-zero (a real upward or downward trend exists).


In [ ]:
DRIFTING_STOCKS = ["IDFCFIRSTB", "ICICIBANK", "IDEA"]

for ticker in DRIFTING_STOCKS:
    sub = shareholding[shareholding["ticker"] == ticker].sort_values("quarter_end_date").reset_index(drop=True)
    sub["quarter_index"] = range(1, len(sub) + 1)

    X = sm.add_constant(sub["quarter_index"])
    y = sub["public_retail_pct"]
    trend_model = sm.OLS(y, X).fit()

    slope = trend_model.params["quarter_index"]
    p_val = trend_model.pvalues["quarter_index"]
    print(f"{ticker}: slope = {slope:+.3f} pct-points/quarter, p = {p_val:.4f}, "
          f"R-sq = {trend_model.rsquared:.3f}")
    print(sub[["quarter_end_date", "public_retail_pct"]].to_string(index=False))
    print()

IDFCFIRSTB: slope = -1.190 pct-points/quarter, p = 0.3149, R-sq = 0.167
quarter_end_date  public_retail_pct
      2024-09-30            29.8100
      2024-12-31            47.5700
      2025-03-31            45.8600
      2025-06-30            43.6400
      2025-09-30            41.8900
      2025-12-31            33.0500
      2026-03-31            32.9300
      2026-06-30            31.7300

ICICIBANK: slope = +2.165 pct-points/quarter, p = 0.0345, R-sq = 0.553
quarter_end_date  public_retail_pct
      2024-09-30             9.3400
      2024-12-31             9.0800
      2025-03-31             9.1200
      2025-06-30             9.0600
      2025-09-30             9.1100
      2025-12-31             9.1300
      2026-03-31            25.3600
      2026-06-30            23.6800

IDEA: slope = -1.773 pct-points/quarter, p = 0.0049, R-sq = 0.758
quarter_end_date  public_retail_pct
      2024-09-30            21.9400
      2024-12-31            25.0500
      2025-03-31            23.55

**Reading the result:** a significant positive slope (p < 0.05) confirms a real upward drift in retail ownership for that stock.

IDEA : the clean case. Steady, monotonic decline every single quarter (21.9% → 13.1%), R²=0.758 means a straight line explains 76% of the variance.

ICICIBANK : technically significant, but can't call it a "trend." Looking at the actual numbers, flat at 9.1% for 6 straight quarters, then a sudden jump to 25.36% and 23.68% in the last two. That's not drift — it's a step-change / regime shift

IDFCFIRSTB : No statistically detectable trend at all (p=0.315). This is the opposite of what the earlier EDA flagged.

## Test 8 — Did behavior differ by calendar year (macro/regime effect)?

**Question:** Is null result stable across 2024, 2025, and 2026, or is it being driven by one unusual year?

**Why Kruskal-Wallis:** same reasoning as Test 3  more than 2 groups (3 calendar years), data is skewed, so the non-parametric >2-group test is appropriate over ANOVA.

**H0:** The distribution of each metric is the same across all 3 calendar years.

**H1:** At least one year differs from the others.


In [ ]:
metrics_df["result_year"] = metrics_df["results_date"].dt.year

print(metrics_df.groupby(["result_year", "window_days"]).size(), "\n")

for window in [3, 5]:
    sub = metrics_df[metrics_df["window_days"] == window]
    print(f"--- window = {window} ---")
    for metric in ["volume_spike_ratio", "price_range_pct", "realized_vol"]:
        groups = [g[metric].dropna().values for _, g in sub.groupby("result_year") if len(g) > 0]
        stat, p = kruskal(*groups)
        medians_by_year = sub.groupby("result_year")[metric].median()
        print(f"{metric}: H = {stat:.4f}, p = {p:.4f}")
        print(f"  medians by year: {medians_by_year.to_dict()}")
    print()

result_year  window_days
2024         3              20
             5              20
2025         3              80
             5              80
2026         3              60
             5              60
dtype: int64 

--- window = 3 ---
volume_spike_ratio: H = 2.2497, p = 0.3247
  medians by year: {2024: 1.2236366913771004, 2025: 1.3354376937261367, 2026: 1.2353864317353627}
price_range_pct: H = 7.5269, p = 0.0232
  medians by year: {2024: 1.1566667376492914, 2025: 1.2493315816498995, 2026: 1.1034251109201119}
realized_vol: H = 4.1181, p = 0.1276
  medians by year: {2024: 0.024661954708884423, 2025: 0.020466209391328975, 2026: 0.01958210643459779}

--- window = 5 ---
volume_spike_ratio: H = 2.9830, p = 0.2250
  medians by year: {2024: 1.040798748274789, 2025: 1.2094558891253806, 2026: 1.1701092888586755}
price_range_pct: H = 8.6973, p = 0.0129
  medians by year: {2024: 1.045410380941945, 2025: 1.1711655154084282, 2026: 1.0698669196293382}
realized_vol: H = 2.8335, p = 0.2425
  

**Reading the result:** p < 0.05 on any metric -> that metric's typical value genuinely shifted across years.

2025 stands out noticeably higher than both 2024 and 2026 at both windows.

but , we can come to any conclusion from this because , 2024 has only 20 observations per window vs. 80 (2025) and 60 (2026). **sample is unbalanced across years**